# Registering your own optimizer

`l2co-optimizers` ships with 60+ built-in optimizers from [`optax`](https://github.com/google-deepmind/optax) and [`evosax`](https://github.com/RobertTLange/evosax), plus SHADE, TuRBO, an RBF trust region, L-BFGS and random search, all referenced by string name through an [`OptimizationStep`](api.md/#l2co_optimizers.OptimizationStep). This guide shows how to **port your own optimizer** so it can be referenced the same way, and driven by [`l2co`](https://github.com/bessagroup/l2co)'s `RolloutWrapper` and the experiment pipelines exactly like a built-in one.

There are three steps:

1. Write a **factory** that returns an [`UpdateClass`](api.md/#l2co_optimizers.UpdateClass).
2. **Register** it under a name with [`register_optimizer`](api.md/#l2co_optimizers.register_optimizer).
3. Reference that name from an [`OptimizationStep`](api.md/#l2co_optimizers.OptimizationStep).

This is the same extension point that [`l2co`](https://github.com/bessagroup/l2co) and [`rl2co`](https://github.com/bessagroup/rl2co) use to inject their meta-optimizers into the registry, without `l2co-optimizers` ever depending on them.


## How optimizers are resolved

Every optimizer name lives in a registry: a plain dictionary mapping `name -> factory`. When a run is built (by l2co's `init_run_state`, which its `RolloutWrapper` calls), the name is looked up via [`optimizer_mapping`](api.md/#l2co_optimizers.optimizer_mapping) and the factory is called to obtain an [`UpdateClass`](api.md/#l2co_optimizers.UpdateClass):

```text
OptimizationStep(optimizer='myopt', ...)
        |
        v  optimizer_mapping('myopt')
   factory(**hyperparameters, model, loss_fn, pass_rng, opt_hash, bounded, stop_fn)
        |
        v
   UpdateClass(init_fn, step_fn, popsize, hash, stop_fn)
        |
        v  stepped by UpdateClass.init_state / step / run
   loss trajectories (OptHistory per step)
```

### The factory contract

Your factory is always called with this keyword contract:

- **`model`** (a PyTree) -- the parameters to optimize; its inexact-array leaves are the decision variables, the rest is static structure.
- **`loss_fn`** -- `loss_fn(model, **sample)`, or `loss_fn(model, key=key, **sample)` when `pass_rng`, returning a scalar.
- **`pass_rng`** (`bool`) -- whether `loss_fn` takes a `key` (a stochastic loss).

There is no task object: when the problem is an [`l2co_tasks.Task`](https://github.com/bessagroup/l2co-tasks), l2co unpacks its `model`, `loss_fn` and `pass_rng` into these keywords.
- **`opt_hash`** (`int`) -- a stable id stamped into every [`OptHistory`](api.md/#l2co_optimizers.OptHistory) entry (`update_step`) so the trajectory can be traced back to this optimizer configuration.
- **`bounded`** (`tuple[float | None, float | None]`) -- box constraints applied to the parameters after each step.
- **`stop_fn`** (`Callable` or `None`) -- a stopping function derived from `OptimizationStep.stopping_criteria`, `None` for never stop; forward it through to the `UpdateClass` unchanged.
- **`**hyperparameters`** -- everything in `OptimizationStep.hyperparameters` (e.g. `learning_rate`). These are *your* optimizer's knobs.

The factory must return an [`UpdateClass`](api.md/#l2co_optimizers.UpdateClass).

### The `UpdateClass` contract

[`UpdateClass`](api.md/#l2co_optimizers.UpdateClass) is a static (JIT-friendly) `eqx.Module`. The five fields every
optimizer must supply:

| Field | Meaning |
| --- | --- |
| `init_fn` | Builds the initial optimizer state. Signature: `init_fn(params, key=None, **sample) -> opt_state`. |
| `step_fn` | Performs **one** optimization step (see below). |
| `popsize` | Number of candidates evaluated per step (`1` for gradient-based optimizers). |
| `hash` | The `opt_hash` passed to the factory. |
| `stop_fn` | The `stop_fn` passed to the factory; `None` never stops. |

and five optional ones, all defaulted, which let the optimizer take part in
budget accounting and in the state transfer applied when a meta-optimizer
switches away from it and back:

| Field | Meaning |
| --- | --- |
| `max_fevals_per_step` | Worst-case evaluations one step can bill, when that is more than `popsize` (line-search optimizers). Read it through `fevals_bound`. |
| `family` | `'gradient'` (default), `'population'` or `'distribution'` — see below. |
| `ask_fn` | `ask_fn(opt_state, key) -> (params, opt_state)`; the optimizer's own sampling step. Required for `'distribution'`, ignored otherwise. |
| `transfer_read_fn` | `transfer_read_fn(opt_state) -> (sigma, conf)`; projects your state onto the scale a switch carries. |
| `transfer_write_fn` | `transfer_write_fn(bundle, opt_state) -> opt_state`; folds a carried scale back into your state. |

**`step_fn` is where the work happens.** The rollout scan calls it as:

```python
(new_params, new_opt_state, new_key), history = step_fn((params, opt_state, key), sample=sample)
```

- The **carry** is the 3-tuple `(params, opt_state, key)`. `params` always carries a leading `popsize` axis, i.e. shape `(popsize, *param_shape)` on every array leaf.
- `sample` is a batch of the dataset (an empty dict for a data-free problem). Combine the parameters back into a model with `eqx.combine(params, static)` before evaluating the loss.
- It must return the **new carry** plus an [`OptHistory`](api.md/#l2co_optimizers.OptHistory) record for this step.

An [`OptHistory`](api.md/#l2co_optimizers.OptHistory) entry stores, per step:

- `loss` -- shape `(popsize,)`; the loss of the *current* population (before the update).
- `params`, `grads` -- the current population's parameters and gradients (use `jnp.nan` gradients for derivative-free methods).
- `update_step` -- set to `opt_hash`.
- `fevals` -- number of function evaluations this step (`1` for a single gradient step, `popsize` for a population sweep).
- `iterations` -- number of inner iterations this step (usually `1`).

Box constraints are *your* responsibility: clip the new parameters to `bounded` before returning them, mirroring the built-in closure factories in `l2co_optimizers._src.optax_implementations` / `evosax_implementations`. `bounded` may be `None` or carry a `None` side, meaning unbounded.

In [ ]:
import equinox as eqx
import jax
import jax.numpy as jnp
import jax.random as jr

from l2co_optimizers import (
    OptHistory,
    OptimizationStep,
    UpdateClass,
    optimizer_mapping,
    optimizers,
    register_optimizer,
    vmapped_loss_and_grad,
)

## Writing a custom optimizer

As a worked example we implement **heavy-ball (momentum) gradient descent** from scratch -- not via `optax`, so the full contract is visible. The optimizer state is a velocity buffer; each step computes the loss and gradient of the current population, updates the velocity, and takes a step.

We use [`vmapped_loss_and_grad`](api.md/#l2co_optimizers.vmapped_loss_and_grad) to evaluate loss and gradients across the leading `popsize` axis, exactly as the built-in `optax` factory does.

In [ ]:
def heavyball_factory(
    *,
    model,
    loss_fn,
    pass_rng,
    opt_hash,
    bounded=(None, None),
    stop_fn=None,
    learning_rate=1e-2,
    momentum=0.9,
    **kwargs,
):
    """Factory for a hand-rolled heavy-ball (momentum) optimizer.

    Matches the registry contract: ``model``, ``loss_fn``, ``pass_rng``,
    ``opt_hash``, ``bounded`` and ``stop_fn`` are supplied by whoever
    builds the run; ``learning_rate`` and ``momentum`` arrive through
    ``OptimizationStep.hyperparameters``. Stochastic losses
    (``pass_rng=True``) are out of scope for this example.
    """
    del pass_rng
    # Static (non-array) part of the model; recombined with the live
    # parameters inside the step via ``eqx.combine``.
    _, static = eqx.partition(model, eqx.is_inexact_array)

    def init_fn(params, key=None, **sample):
        # One velocity buffer per leaf, shaped like params (popsize, dim).
        return jax.tree.map(jnp.zeros_like, params)

    def step_fn(carry, sample):
        params, velocity, key = carry

        # Loss + gradient of the current population (leading popsize axis).
        loss, grads = vmapped_loss_and_grad(
            eqx.combine(params, static), loss_fn, sample
        )

        # Heavy-ball update.
        velocity = jax.tree.map(lambda v, g: momentum * v + g, velocity, grads)
        new_params = jax.tree.map(
            lambda p, v: p - learning_rate * v, params, velocity
        )

        # Apply box constraints.
        new_params = jax.tree.map(lambda p: jnp.clip(p, *bounded), new_params)

        history = OptHistory(
            loss=loss,
            params=params,
            grads=grads,
            update_step=jnp.array(opt_hash, dtype=int),
            fevals=jnp.array(1, dtype=int),
            iterations=jnp.array(1, dtype=int),
        )

        new_key, _ = jr.split(key)
        return (new_params, velocity, new_key), history

    return UpdateClass(
        init_fn=init_fn,
        step_fn=step_fn,
        popsize=1,  # gradient-based: a single point
        hash=opt_hash,
        stop_fn=stop_fn,
    )

## Registering the optimizer

[`register_optimizer`](api.md/#l2co_optimizers.register_optimizer) inserts the factory into the global registry under a (lowercased) name. After this call, `'heavyball'` resolves through [`optimizer_mapping`](api.md/#l2co_optimizers.optimizer_mapping) like any built-in optimizer.

In a package, you would call `register_optimizer` at import time so that simply importing your package wires the optimizer into the registry.

In [ ]:
register_optimizer("heavyball", heavyball_factory)

# It is now part of the registry and resolvable by name.
print("heavyball" in optimizers)
optimizer_mapping("heavyball")

## Building and stepping it

From here on, the custom optimizer behaves like any built-in one. Build an [`OptimizationStep`](api.md/#l2co_optimizers.OptimizationStep) that references it by name, passing the optimizer-specific knobs through `hyperparameters`. Resolving it is exactly what l2co does when it builds a run: look the name up and call the factory with the registry keywords.

The problem is just those keywords -- here a 10-D shifted sphere.


In [ ]:
def sphere(x, **sample):
    return jnp.sum((x - 0.5) ** 2)


model = jnp.zeros(10)

heavyball_step = OptimizationStep(
    optimizer="heavyball",
    hyperparameters={"learning_rate": 5e-2, "momentum": 0.9},
)

heavyball = optimizer_mapping(heavyball_step.optimizer)(
    model=model,
    loss_fn=sphere,
    pass_rng=False,
    opt_hash=heavyball_step.hash,
    bounded=(-5.0, 5.0),
    stop_fn=heavyball_step.stopping_fn,
    **heavyball_step.hyperparameters,
)

# One leading popsize axis, as the rollout machinery hands it over.
params = jnp.repeat(model[None], heavyball.popsize, axis=0)
carry = (params, heavyball.init_fn(params, jr.key(0)), jr.key(0))
step = jax.jit(heavyball.step_fn)

losses = []
for _ in range(200):
    carry, history = step(carry, sample={})
    losses.append(float(history.loss.min()))

print(f"best loss found: {min(losses):.4g}")

Each step returns an [`OptHistory`](api.md/#l2co_optimizers.OptHistory) record, just like a built-in optimizer. The hand-written loop above is only there to show the contract; the next section runs the same optimizer through the loop every `UpdateClass` carries.


In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots()
ax.plot(losses)
ax.set_yscale("log")
ax.set_xlabel("generation")
ax.set_ylabel("loss")
ax.set_title("heavyball on a shifted sphere (10D)")
fig.tight_layout()

## Running it with the built-in loop

Every [`UpdateClass`](api.md/#l2co_optimizers.UpdateClass) -- yours included, with no extra code -- carries the run loop as methods: `init_state` builds the optimizer state, `run` drives one trajectory as a single `jax.lax.scan`, and `batch_run` drives many realizations at once (`params`, `opt_state` and `key` then carry a leading realization axis). The loop draws its data batches from a [`BatchState`](api.md/#l2co_optimizers.BatchState) over a plain `dict` of arrays; this problem has no data, so the dict is empty. `stop_fn` is checked before every step, and the result reports a per-iteration [`HistoryState`](api.md/#l2co_optimizers.HistoryState) instead of the full population trajectory.

To run it on an `l2co_tasks.Task` -- which builds the population, the batch state and the realizations for you -- hand the same `OptimizationStep` to [`l2co`](https://github.com/bessagroup/l2co)'s `init_run_state` (or its `RolloutWrapper`); no further wiring is needed.

In [ ]:
from l2co_optimizers import BatchState

batch_state = BatchState.init(dataset={}, batch_size=None, key=jr.key(0))
opt_state = heavyball.init_state(params, batch_state, {}, jr.key(0))

_, best_params, best_loss, _, _, history = eqx.filter_jit(heavyball.run)(
    opt_state=opt_state,
    params=params,
    batch_state=batch_state,
    dataset={},
    key=jr.key(1),
    n_iterations=200,
    verbose=False,
)

print(f"best loss found: {float(best_loss):.4g}")
print("per-iteration minimum:", history.output_min.shape)

## Population-based optimizers

The same contract covers derivative-free / population-based optimizers -- only a few details change:

- Set `popsize > 1`. The rollout machinery hands you a population of that size, so `params` (and therefore `loss`) carry a leading axis of length `popsize`.
- Evaluate the whole population at once with [`vmapped_loss`](api.md/#l2co_optimizers.vmapped_loss) (no gradients needed), and store `grads` as `jnp.nan` of the right shape in the [`OptHistory`](api.md/#l2co_optimizers.OptHistory).
- Set `fevals` to `popsize` (one evaluation per population member).

For a fully worked example, see the EvoSax closure factories in `l2co_optimizers._src.evosax_implementations` (`evosax_distribution_based_fn` / `evosax_population_based_fn`), and `l2co_optimizers._src.random_search` for the one-shot random-search variant.

## Taking part in state transfer

Everything above is enough to *run* your optimizer. One more step lets it
take part in a **switch**: when a meta-optimizer moves from another
optimizer to yours, the handshake can hand over the scale of the search so far,
instead of your optimizer restarting at whatever its constructor chose.
On a menu whose members' initial scales differ by three orders of
magnitude, that gap is the difference between resuming a converged search
and starting over.

Declare it with `build_transfer_fns`, which returns the reader/writer
pair for your optimizer's family:

```python
from l2co_optimizers import FAMILY_GRADIENT, build_transfer_fns

read_fn, write_fn = build_transfer_fns(
    'heavyball', FAMILY_GRADIENT, hyperparameters
)

return UpdateClass(
    init_fn=init_fn,
    step_fn=step_fn,
    popsize=1,
    hash=opt_hash,
    stop_fn=stop_fn,
    family=FAMILY_GRADIENT,
    transfer_read_fn=read_fn,
    transfer_write_fn=write_fn,
)
```

Pass `hyperparameters` through: the pair is built at construction time
precisely because it needs values your state does not retain — the
learning rate, any clip bounds, and (for evosax algorithms) the ravel
function used to flatten a solution pytree.

Which `family` you declare decides what a switching caller does with the
population, so it is not cosmetic:

- **`'gradient'`** — unit population. The caller hands you the best point
  so far. Heavy-ball is this case.
- **`'population'`** — your stored population *is* your state; the caller
  hands you a population with the best point injected.
- **`'distribution'`** — you hold an explicit search distribution. The
  caller writes the location and scale into your state and then calls your
  `ask_fn` so **you** draw the population. If your `step_fn` caches
  anything from its own sampling (noise draws, perturbations) and reads it
  back on the next update, you *must* be this family and you *must*
  supply `ask_fn` — otherwise a caller substitutes a population your
  cached draws do not correspond to, and your adaptation silently
  proceeds against a fiction. See l2co ADR 0012.

Declare `family` explicitly rather than letting it default; the default
is `'gradient'` and would misdescribe anything else.

If your optimizer holds no notion of scale, pass nothing: the defaults
report "no information" and every writer skips.


## Summary

To port your own optimizer:

1. **Write a factory** `f(*, model, loss_fn, pass_rng, opt_hash, bounded, stop_fn, **hyperparameters) -> UpdateClass`.
2. Inside it, define `init_fn` (build the optimizer state) and `step_fn` (one step: evaluate -> update -> emit an [`OptHistory`](api.md/#l2co_optimizers.OptHistory), clipping to `bounded`).
3. **Declare `family`** and, if it holds a scale, wire
   `transfer_read_fn` / `transfer_write_fn` via `build_transfer_fns` —
   plus `ask_fn` if it draws its own post-switch population.
4. **Register** it with `register_optimizer('myopt', f)` -- at import time if you ship it in a package.
5. Reference it by name from an [`OptimizationStep`](api.md/#l2co_optimizers.OptimizationStep), passing knobs through `hyperparameters`.

Because the optimizer is now in the registry, [`l2co`](https://github.com/bessagroup/l2co)'s `RolloutWrapper` and `RunOptimization` pipeline, the meta-optimizers' menus and the full experiment tooling all pick it up -- no further wiring required.
